# 01 — Data Collection

| Source | Data | Years |
|---|---|---|
| **CollegeFootballData API** | Game results + box score stats (yards, pass/rush, turnovers) | 2005–2024 |
| **School athletic sites** | Rosters (height, weight, position) | 2014–2024 |


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
TEST_MODE = True   # True = 2021-2024 only (~65 API calls, ~5 min)
                   # False = full 2005-2024 (~320 calls, ~25 min)
START_YEAR   = 2021 if TEST_MODE else 2005
END_YEAR     = 2024
ROSTER_START = 2021 if TEST_MODE else 2014
print(f'Mode: {"TEST (2021-2024)" if TEST_MODE else "FULL (2005-2024)"}')
print(f'Games+BoxScores: {START_YEAR}–{END_YEAR}  |  Rosters: {ROSTER_START}–{END_YEAR}')

In [ ]:
import sys
sys.path.insert(0, '..')
from src.scraper import IVY_CFBD_NAMES
from src.roster_scraper import SCHOOL_SITES
print('CFBD schools:', list(IVY_CFBD_NAMES.keys()))

## Step 1 — Game results + box score stats via cfbd API

In [ ]:
from src.scraper import scrape_all
scrape_all(start_year=START_YEAR, end_year=END_YEAR, data_dir='../data/raw')

## Step 2 — Rosters from school athletic sites

In [ ]:
from src.roster_scraper import scrape_all_rosters
rosters = scrape_all_rosters(start_year=ROSTER_START, end_year=END_YEAR, data_dir='../data/raw')
print(rosters.groupby(['school','year']).size().reset_index(name='players').to_string())

## Step 3 — Build season stats (game results + box scores merged)

In [ ]:
import pandas as pd
from pathlib import Path
from src.features import build_stats_from_games

schedules = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
stats = build_stats_from_games(schedules)

# Merge in box score stats if available
box_path = Path('../data/raw/team_stats/box_stats_raw.csv')
if box_path.exists():
    box = pd.read_csv(box_path)
    box['school'] = box['school'].str.lower()
    stats = stats.merge(box, on=['school','year'], how='left')
    print(f'Box score stats merged: {box.shape[0]} rows, {box.shape[1]} cols')
else:
    print('No box stats yet — will appear after Step 1 runs with new scraper')

stats.to_csv('../data/raw/team_stats/team_stats_raw.csv', index=False)
print('Stats:', stats.shape)
show_cols = ['school','year','win_pct','ivy_win_pct'] + [c for c in ['totalYards','rushingYards','netPassingYards','turnovers'] if c in stats.columns]
stats[show_cols].head(8)

## Summary

In [ ]:
import pandas as pd

rosters   = pd.read_csv('../data/raw/rosters/rosters_raw.csv')
schedules = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
stats     = pd.read_csv('../data/raw/team_stats/team_stats_raw.csv')

print(f'Games  : {len(schedules):,} rows | {schedules.school.nunique()} schools | {schedules.year.min()}–{schedules.year.max()}')
print(f'Rosters: {len(rosters):,} rows  | {rosters.school.nunique()} schools | {rosters.year.min()}–{rosters.year.max()}')
print(f'Stats  : {len(stats):,} rows   | {stats.columns.tolist()}')
print()
display(stats.groupby('school')['year'].agg(['min','max','count']))
print()
box_cols = [c for c in stats.columns if c in ['totalYards','rushingYards','netPassingYards','turnovers','yardsPerPass']]
if box_cols:
    print('Box score coverage (non-null totalYards):')
    display(stats.pivot_table(index='school', columns='year', values='totalYards', aggfunc='count').fillna(0).astype(int))